In [ ]:
# Advanced Hyperparameter tuning with bayesian optimization

In [2]:
# Introduction to Bayesian Optimization

# Bayesian Optimization:
    # -Advanced method for hyperparameter tuning that balances exploration(searching new regions) and exploitation(refining promising regions)
    # -Uses a probabilistic model to guide the search for optimal hyperparameters 

    # Working:
        # Surrogate Model:
            # -Builds a probabilistic model(eg: Guassian process) of the objective function based on prior evaluations
        # Acquistion Function:
            # -balances exploration and exploitation by choosing the next hyperparameter to evaluate based on predcited performance and uncertainty.
        # Iterarive refinement
            # -Updates the surrogate model after each evaluation, refining the search 

    # Use of Bayesian optimization:
        # -efficient for high-dimensional and expensive-to-evaluate functions
        # -reduces the number of evaluations required to find near-optimal hyperparameters 


In [3]:
# Using libraries for bayesian optimization
# Popular Libaries
    # Hyperopt:
        # -Simplifies Bayesian optimization for hyperparameter tuning
        # -works with "fmin" to minimize objective function over a parameter space
    # Optuna:
        # -Flexible and user-friendly library for hyperparameter optimization
        # -Supports dynamic search spaces and pruning of unpromising trials 

In [4]:
# Understanding Exploration vs Exploitation

# Exploration:
    # -Focuses on sampling hyperparameters from unexploration
    # -useful for identifying new areas of high potential
# Exploitation:
    # -Focuses on refining the search around regions with known high performance
    # -Useful for fine-tuning near-optimal hyperparameters 
# Bayesian optimization's Advantages:
    # -Balances these approaches using the acquistion function to minimize unnecessary evaluations while improving results

In [5]:
# Exercise:
    # Apply bayesian optimization using Optuna to tune a XGBoost model and Compare the rsults with Grid Search and Random Search

In [20]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import optuna 

# Load the dataset
data = load_breast_cancer()
X, y =data.data, data.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Strandardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

print(f"Training data shape: {X_train.shape}")
print(f"Training data shape: {X_test.shape}")

# train a baseline XGBoost model
baseline_model = XGBClassifier(eval_metric='logloss', random_state=42)
baseline_model.fit(X_train, y_train)

# Evaluate the model
baseline_pred = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"baseline XGBoost Accuracy: {baseline_accuracy: .4f}")

# bayesian
# Define the objective function for optuna
def objective(trail):
    params = {
        'n_estimators' : trail.suggest_int('n_estimators',50,500),
        'max_depth':trail.suggest_int('max_depth',3,100),
        'learning_rate' : trail.suggest_float('learning_rate', 0.01,0.3),
        'subsample' : trail.suggest_float('subsample', 0.6,1.0),
        'Colsample_bytree' : trail.suggest_float('Colsample_bytree', 0.6,1.0),
        'gamma' : trail.suggest_float('gamma', 0, 5),
        'reg_alpha' : trail.suggest_float('reg_alpha', 0, 10),
        'reg_lambda' : trail.suggest_float('reg_lambda', 0, 10)
    }

    # train XGBoost model with suggested params
    model = XGBClassifier(eval_metric='logloss', random_state=42, **params)
    model.fit(X_train, y_train)\
    
    # Evaluate the model
    pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    return accuracy

# create a optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# best hyperparameters
print("best hyperparameters: ", study.best_params)
print("best accuracy: ", study.best_value)


# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0]
}

# Train XGBoost with Grid Search
grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss',random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best parameters and accuracy
print("\n\n\nGrid Search Best Parameters: ", grid_search.best_params_)
print("Grid Search Best Accuracy:", grid_search.best_score_)

# Define parameter distributions
param_dist = {
    'n_estimators': [50,100,200,300,400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

# Train XGBoost with Random Search
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=3,
    verbose=1,
    random_state=42
)

random_search.fit(X_train, y_train)

# Best parameters and accuracy
print("\n\n\nRandom Search Best Parameters:", random_search.best_params_)
print("Random Search Best Accuracy:", random_search.best_score_)


# Baseline XGBoost Accuracy: 0.9561
# Optuna Best Accuracy:  0.9649122807017544
# Grid Search Best Accuracy: 0.9757900546067155
# Random Search Best Accuracy: 0.9758045776693388

[I 2026-02-13 09:40:39,510] A new study created in memory with name: no-name-fc50f85d-99ae-4ff9-9b5b-b5776749d4aa
[I 2026-02-13 09:40:39,601] Trial 0 finished with value: 0.9649122807017544 and parameters: {'n_estimators': 68, 'max_depth': 93, 'learning_rate': 0.07003668596357399, 'subsample': 0.8552209358068288, 'Colsample_bytree': 0.7453656914639331, 'gamma': 0.9758035902753748, 'reg_alpha': 8.661575260512892, 'reg_lambda': 2.019962699524449}. Best is trial 0 with value: 0.9649122807017544.


Training data shape: (455, 30)
Training data shape: (114, 30)
baseline XGBoost Accuracy:  0.9649


[I 2026-02-13 09:40:39,731] Trial 1 finished with value: 0.9736842105263158 and parameters: {'n_estimators': 241, 'max_depth': 100, 'learning_rate': 0.11217868937429029, 'subsample': 0.6887054541252001, 'Colsample_bytree': 0.9131202427383237, 'gamma': 0.7159725594576749, 'reg_alpha': 5.578924290668104, 'reg_lambda': 1.7130137612809648}. Best is trial 1 with value: 0.9736842105263158.
[I 2026-02-13 09:40:39,891] Trial 2 finished with value: 0.956140350877193 and parameters: {'n_estimators': 365, 'max_depth': 69, 'learning_rate': 0.07455263338419833, 'subsample': 0.8248482373497398, 'Colsample_bytree': 0.7334457089801573, 'gamma': 4.076209935195217, 'reg_alpha': 9.153278794882372, 'reg_lambda': 5.99208743418099}. Best is trial 1 with value: 0.9736842105263158.
[I 2026-02-13 09:40:39,969] Trial 3 finished with value: 0.956140350877193 and parameters: {'n_estimators': 95, 'max_depth': 77, 'learning_rate': 0.1069000807982614, 'subsample': 0.6170443058281483, 'Colsample_bytree': 0.8972713802

best hyperparameters:  {'n_estimators': 241, 'max_depth': 100, 'learning_rate': 0.11217868937429029, 'subsample': 0.6887054541252001, 'Colsample_bytree': 0.9131202427383237, 'gamma': 0.7159725594576749, 'reg_alpha': 5.578924290668104, 'reg_lambda': 1.7130137612809648}
best accuracy:  0.9736842105263158
Fitting 3 folds for each of 81 candidates, totalling 243 fits



Grid Search Best Parameters:  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.6}
Grid Search Best Accuracy: 0.9735970721505751
Fitting 3 folds for each of 50 candidates, totalling 150 fits



Random Search Best Parameters: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
Random Search Best Accuracy: 0.975775531544092
